# Chapter 9 — Time Series Analysis & Forecasting: Network Traffic

## 📋 Latihan Soal

### Dataset
- **Mirai Botnet Dataset** — https://huggingface.co/datasets/bornpresident/mirai_botnet
- 1.083.206 baris network traffic dengan timestamp implisit (urutan capture)
- Sample 10000 baris untuk time series analysis

### Teknik
Time series decomposition, trend analysis, seasonality detection, rolling statistics, anomaly detection, autocorrelation, basic forecasting (moving average, exponential smoothing)

In [ ]:
# ============================================
# SETUP: Load Mirai Botnet Dataset for Time Series
# ============================================
# Dataset: https://huggingface.co/datasets/bornpresident/mirai_botnet
from datasets import load_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
ds = load_dataset('bornpresident/mirai_botnet')
df = ds['train'].to_pandas()
df.columns = [c.replace(' ', '_') for c in df.columns]
df = df.rename(columns={'label': 'traffic_type'})
df['traffic_type'] = df['traffic_type'].apply(
    lambda x: 'Benign' if x == 'BenignTraffic' else 'Malicious'
)

# Buat timestamp simulasi (dataset asli tidak punya timestamp)
# Asumsi: setiap baris = 1 detik capture berurutan
df['timestamp'] = pd.date_range('2024-01-01', periods=len(df), freq='1s')
df = df.set_index('timestamp').sort_index()

# Sample 10000 baris untuk time series analysis
np.random.seed(42)
sample_size = 10000
# Ambil contiguous block untuk menjaga urutan waktu
start_idx = np.random.randint(0, len(df) - sample_size)
df_ts = df.iloc[start_idx:start_idx + sample_size].copy()

# Resample ke 10-second windows untuk smoothing
df_resampled = df_ts.resample('10s').agg({
    'Rate': 'mean',
    'Tot_size': 'sum',
    'Duration': 'mean',
    'AVG': 'mean',
    'syn_flag_number': 'sum',
    'traffic_type': lambda x: 'Malicious' if 'Malicious' in x.values else 'Benign',
}).dropna()

print(f"Dataset asli: {len(df)} baris")
print(f"Sample: {len(df_ts)} baris ({df_ts.index[0]} → {df_ts.index[-1]})")
print(f"Resampled (10s windows): {len(df_resampled)} baris")
print(f"\nKolom time series: {list(df_resampled.columns)}")
print(f"\nMissing values:")
print(df_resampled.isnull().sum())

---
## Soal 1: Time Series Plot — Visualisasi Traffic Rate

Plot `Rate` sebagai time series. Tambahkan rolling mean (window=50) untuk melihat trend.

In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot utama: Rate over time
axes[0].plot(df_resampled.index, df_resampled['Rate'].values,
             alpha=0.6, linewidth=0.8, label='Rate', color='steelblue')

# Rolling mean
rolling = df_resampled['Rate'].rolling(window=50, center=True).mean()
axes[0].plot(rolling.index, rolling.values,
             color='red', linewidth=2, label='Rolling Mean (window=50)')

axes[0].set_title('Packet Rate Over Time')
axes[0].set_ylabel('Rate (pkts/s)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Zoom: 200 data points pertama
zoom = df_resampled.head(200)
axes[1].plot(zoom.index, zoom['Rate'].values,
             alpha=0.7, linewidth=1.2, marker='o', markersize=3)
axes[1].set_title('Zoom: 200 Data Points Pertama')
axes[1].set_ylabel('Rate (pkts/s)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/ch9_timeseries.png', dpi=150)
print("✅ Time series plot saved to /tmp/ch9_timeseries.png")

---
## Soal 2: Rolling Statistics — Mean, Std, Min, Max

Hitung rolling statistics (window=100) untuk `Rate` dan `Tot_size`. Plot hasilnya.

In [ ]:

window = 100
df_rolling = pd.DataFrame({
    'Rate_mean': df_resampled['Rate'].rolling(window).mean(),
    'Rate_std': df_resampled['Rate'].rolling(window).std(),
    'Rate_min': df_resampled['Rate'].rolling(window).min(),
    'Rate_max': df_resampled['Rate'].rolling(window).max(),
})

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
stats = [('Rate_mean', 'Rolling Mean', 'blue'),
         ('Rate_std', 'Rolling Std Dev', 'orange'),
         ('Rate_min', 'Rolling Min', 'green'),
         ('Rate_max', 'Rolling Max', 'red')]

for ax, (col, title, color) in zip(axes.flat, stats):
    ax.plot(df_rolling.index, df_rolling[col].values, color=color, linewidth=1.2)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Rolling Statistics (window={window})', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/ch9_rolling.png', dpi=150)
print("✅ Rolling statistics saved to /tmp/ch9_rolling.png")

---
## Soal 3: Time Series Decomposition — Trend, Seasonality, Residual

Decompose time series `Rate` menjadi 3 komponen: trend, seasonality, dan residual.

In [ ]:

# Install: pip install statsmodels
try:
    from statsmodels.tsa.seasonal import seasonal_decompose

    # Decompose dengan period 60 (asumsi cycle per menit)
    rate_series = df_resampled['Rate'].dropna()
    decomposition = seasonal_decompose(rate_series, model='additive', period=60)

    fig, axes = plt.subplots(4, 1, figsize=(14, 10))
    
    axes[0].plot(decomposition.observed.index, decomposition.observed.values,
                 color='steelblue', linewidth=0.8)
    axes[0].set_title('Observed')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(decomposition.trend.index, decomposition.trend.values,
                 color='red', linewidth=1.5)
    axes[1].set_title('Trend')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(decomposition.seasonal.index, decomposition.seasonal.values,
                 color='green', linewidth=0.8)
    axes[2].set_title('Seasonal (period=60)')
    axes[2].grid(True, alpha=0.3)

    axes[3].plot(decomposition.resid.index, decomposition.resid.values,
                 color='gray', linewidth=0.5)
    axes[3].set_title('Residual')
    axes[3].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/tmp/ch9_decomposition.png', dpi=150)
    print("✅ Decomposition saved to /tmp/ch9_decomposition.png")

    # Statistik residual
    resid = decomposition.resid.dropna()
    print(f"\n=== Statistik Residual ===")
    print(f"  Mean: {resid.mean():.4f}")
    print(f"  Std:  {resid.std():.4f}")
    print(f"  Min:  {resid.min():.4f}")
    print(f"  Max:  {resid.max():.4f}")

except ImportError:
    print("⚠️ statsmodels tidak terinstall")
    print("  Install: pip install statsmodels")
    # Fallback: manual trend
    fig, axes = plt.subplots(3, 1, figsize=(14, 8))
    rate = df_resampled['Rate'].dropna()
    trend = rate.rolling(window=60, center=True).mean()
    seasonal = rate - trend
    
    axes[0].plot(rate.index, rate.values, color='steelblue', linewidth=0.5)
    axes[0].set_title('Observed')
    axes[1].plot(trend.index, trend.values, color='red', linewidth=1.5)
    axes[1].set_title('Trend (Rolling Mean)')
    axes[2].plot(seasonal.index, seasonal.values, color='gray', linewidth=0.5)
    axes[2].set_title('Seasonal + Residual')
    for ax in axes: ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/tmp/ch9_decomposition_fallback.png', dpi=150)
    print("✅ Fallback decomposition saved")

---
## Soal 4: Autocorrelation (ACF & PACF)

Hitung dan plot ACF (Autocorrelation Function) dan PACF (Partial Autocorrelation) untuk `Rate`. Identifikasi lag yang signifikan.

In [ ]:

# Install: pip install statsmodels
try:
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

    rate_series = df_resampled['Rate'].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    plot_acf(rate_series, lags=100, ax=axes[0])
    axes[0].set_title('ACF — Autocorrelation Function')

    plot_pacf(rate_series, lags=100, ax=axes[1], method='yw')
    axes[1].set_title('PACF — Partial Autocorrelation Function')

    plt.tight_layout()
    plt.savefig('/tmp/ch9_acf_pacf.png', dpi=150)
    print("✅ ACF & PACF saved to /tmp/ch9_acf_pacf.png")

    # Cari lag signifikan
    from statsmodels.tsa.stattools import acf
    acf_values = acf(rate_series, nlags=50)
    significant_lags = np.where(np.abs(acf_values[1:]) > 0.1)[0] + 1
    print(f"\n=== Lag Signifikan (|ACF| > 0.1) ===")
    for lag in significant_lags[:10]:
        print(f"  Lag {lag}: ACF = {acf_values[lag]:.4f}")

except ImportError:
    print("⚠️ statsmodels tidak terinstall")
    print("  Install: pip install statsmodels")
    # Fallback: manual ACF
    rate = df_resampled['Rate'].dropna().values
    rate = (rate - rate.mean()) / rate.std()
    max_lag = 50
    acf_vals = [np.correlate(rate, rate, mode='full')[len(rate)-1+lag] / len(rate)
                for lag in range(max_lag + 1)]

    plt.figure(figsize=(10, 4))
    plt.bar(range(max_lag + 1), acf_vals, color='steelblue', alpha=0.7)
    plt.axhline(0.1, color='red', linestyle='--', alpha=0.5, label='Threshold 0.1')
    plt.axhline(-0.1, color='red', linestyle='--', alpha=0.5)
    plt.title('Manual ACF')
    plt.xlabel('Lag')
    plt.ylabel('Autocorrelation')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/tmp/ch9_acf_fallback.png', dpi=150)
    print("✅ Fallback ACF saved")

---
## Soal 5: Anomaly Detection — Z-Score Method

Deteksi anomali pada time series `Rate` menggunakan Z-score. Threshold: |Z| > 3. Plot anomali pada time series.

In [ ]:

# Z-score anomaly detection
rate = df_resampled['Rate'].dropna()
rolling_mean = rate.rolling(window=50, center=True).mean()
rolling_std = rate.rolling(window=50, center=True).std()

z_scores = (rate - rolling_mean) / rolling_std
anomalies = df_resampled.iloc[np.where(np.abs(z_scores.dropna()) > 3)[0]]

print(f"=== Anomaly Detection (Z-score > 3) ===")
print(f"Total data points: {len(rate)}")
print(f"Anomalies detected: {len(anomalies)} ({len(anomalies)/len(rate)*100:.2f}%)")
if len(anomalies) > 0:
    print(f"\nAnomaly stats:")
    print(f"  Mean Rate: {anomalies['Rate'].mean():.2f} (vs normal: {rate.mean():.2f})")
    print(f"  Max Rate:  {anomalies['Rate'].max():.2f}")
    print(f"  Min Rate:  {anomalies['Rate'].min():.2f}")

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(rate.index, rate.values, alpha=0.5, linewidth=0.5, label='Rate', color='steelblue')
ax.plot(rolling_mean.index, rolling_mean.values, color='red', linewidth=1.5, label='Rolling Mean')

if len(anomalies) > 0:
    ax.scatter(anomalies.index, anomalies['Rate'].values,
               color='red', s=50, zorder=5, label=f'Anomalies ({len(anomalies)})', edgecolors='darkred')

ax.set_title('Anomaly Detection — Z-Score Method (threshold=3)')
ax.set_ylabel('Rate (pkts/s)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/ch9_anomaly.png', dpi=150)
print("\n✅ Anomaly detection saved to /tmp/ch9_anomaly.png")

---
## Soal 6: Moving Average Forecasting

Implementasi forecasting dengan Simple Moving Average (SMA) dan Exponential Moving Average (EMA). Bandingkan akurasinya.

In [ ]:

# Split train/test (80/20)
split = int(len(df_resampled) * 0.8)
train = df_resampled['Rate'].iloc[:split]
test = df_resampled['Rate'].iloc[split:]

# Simple Moving Average (window=50)
window = 50
sma_forecast = pd.Series([train.iloc[-window:].mean()] * len(test), index=test.index)

# Exponential Moving Average (alpha=0.1)
alpha = 0.1
ema_forecast = pd.Series([train.iloc[-1]] * len(test), index=test.index)
ema_prev = train.iloc[-1]
for i in range(len(test)):
    ema_prev = alpha * train.iloc[-1] + (1 - alpha) * ema_prev
    ema_forecast.iloc[i] = ema_prev

# Hitung error (MAE)
sma_mae = np.mean(np.abs(test - sma_forecast))
ema_mae = np.mean(np.abs(test - ema_forecast))
print(f"=== Forecasting Accuracy (MAE) ===")
print(f"  SMA (window={window}):  {sma_mae:.4f}")
print(f"  EMA (alpha={alpha}):    {ema_mae:.4f}")
print(f"  Lebih baik: {'EMA' if ema_mae < sma_mae else 'SMA'}")

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train.index, train.values, label='Train', color='steelblue', linewidth=0.8)
ax.plot(test.index, test.values, label='Test (Actual)', color='green', linewidth=0.8)
ax.plot(sma_forecast.index, sma_forecast.values, label=f'SMA (window={window})', color='orange', linewidth=1.5)
ax.plot(ema_forecast.index, ema_forecast.values, label=f'EMA (alpha={alpha})', color='red', linewidth=1.5)
ax.axvline(test.index[0], color='gray', linestyle='--', alpha=0.5)
ax.text(test.index[0], ax.get_ylim()[1]*0.9, 'Train/Test Split', fontsize=10, ha='center')
ax.set_title('Moving Average Forecasting: SMA vs EMA')
ax.set_ylabel('Rate (pkts/s)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/ch9_forecast_ma.png', dpi=150)
print("\n✅ Forecasting MA saved to /tmp/ch9_forecast_ma.png")

---
## Soal 7: Seasonal Analysis — Pola per Waktu

Analisis pola traffic berdasarkan waktu: jam, menit. Apakah ada pattern musiman dalam data network traffic?

In [ ]:

df_ts_copy = df_ts.copy()
df_ts_copy['hour'] = df_ts_copy.index.hour
df_ts_copy['minute'] = df_ts_copy.index.minute
df_ts_copy['second'] = df_ts_copy.index.second

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Pola per jam (dalam sample)
hourly = df_ts_copy.groupby('hour')['Rate'].agg(['mean', 'std', 'count'])
axes[0].bar(hourly.index, hourly['mean'], yerr=hourly['std'], capsize=3, alpha=0.7, color='steelblue')
axes[0].set_title('Rata-rata Rate per Jam (dalam sample)')
axes[0].set_xlabel('Jam')

# Pola per menit
minutely = df_ts_copy.groupby('minute')['Rate'].mean()
axes[1].plot(minutely.index, minutely.values, color='coral', linewidth=1.5)
axes[1].set_title('Rata-rata Rate per Menit')
axes[1].set_xlabel('Menit')

# Pola per detik
secondly = df_ts_copy.groupby('second')['Rate'].mean()
axes[2].plot(secondly.index, secondly.values, color='teal', linewidth=1.2)
axes[2].set_title('Rata-rata Rate per Detik')
axes[2].set_xlabel('Detik')

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle('Seasonal Analysis — Pola Waktu Network Traffic', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/ch9_seasonal.png', dpi=150)
print("✅ Seasonal analysis saved to /tmp/ch9_seasonal.png")

# Statistik per traffic type
print("\n=== Rate per Traffic Type (waktu) ===")
type_time = df_ts_copy.groupby(['traffic_type', 'hour'])['Rate'].mean().unstack()
print(type_time.round(2))

---
## Soal 8: Cross-Correlation — Hubungan Antar Fitur Waktu

Hitung cross-correlation antara `Rate` dan `Tot_size`. Apakah ada lag di mana korelasi paling kuat?

In [ ]:

# Cross-correlation Rate vs Tot_size
rate = df_resampled['Rate'].dropna()
tot_size = df_resampled['Tot_size'].dropna()

# Normalisasi
rate_norm = (rate - rate.mean()) / rate.std()
size_norm = (tot_size - tot_size.mean()) / tot_size.std()

# Cross-correlation pada berbagai lag
max_lag = 100
lags = range(-max_lag, max_lag + 1)
cross_corr = [np.correlate(rate_norm.values, np.roll(size_norm.values, lag), mode='valid')[0] / len(rate_norm)
              for lag in lags]

best_lag = lags[np.argmax(np.abs(cross_corr))]
best_corr = cross_corr[np.argmax(np.abs(cross_corr))]
print(f"=== Cross-Correlation Rate vs Tot_size ===")
print(f"  Lag optimal: {best_lag}")
print(f"  Korelasi:    {best_corr:.4f}")
if abs(best_corr) > 0.5:
    print(f"  → Korelasi {'positif' if best_corr > 0 else 'negatif'} yang kuat")
elif abs(best_corr) > 0.3:
    print(f"  → Korelasi {'positif' if best_corr > 0 else 'negatif'} yang moderat")
else:
    print(f"  → Korelasi lemah")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(lags), cross_corr, color='steelblue', linewidth=1.2)
axes[0].axvline(best_lag, color='red', linestyle='--', label=f'Best lag: {best_lag}')
axes[0].set_title('Cross-Correlation: Rate vs Tot_size')
axes[0].set_xlabel('Lag')
axes[0].set_ylabel('Cross-Correlation')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter: Rate vs Tot_size
axes[1].scatter(rate.values, tot_size.values, alpha=0.3, s=10, color='teal')
axes[1].set_title('Rate vs Tot_size')
axes[1].set_xlabel('Rate')
axes[1].set_ylabel('Tot_size')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/ch9_crosscorr.png', dpi=150)
print("\n✅ Cross-correlation saved to /tmp/ch9_crosscorr.png")

---
## Soal 9: Change Point Detection — Deteksi Pergeseran Pola

Deteksi change point (titik pergeseran pola) pada time series `Rate`. Gunakan metode cumulative sum (CUSUM) atau rolling statistics comparison.

In [ ]:

# Change point detection dengan rolling mean comparison
rate = df_resampled['Rate'].dropna().values
n = len(rate)

# CUSUM (Cumulative Sum)
mean_val = np.mean(rate)
cusum_pos = np.zeros(n)
cusum_neg = np.zeros(n)
threshold = 5 * np.std(rate)

for i in range(1, n):
    cusum_pos[i] = max(0, cusum_pos[i-1] + rate[i] - mean_val - 0.5 * np.std(rate))
    cusum_neg[i] = max(0, cusum_neg[i-1] - rate[i] + mean_val - 0.5 * np.std(rate))

# Deteksi change points
change_points_pos = np.where(cusum_pos > threshold)[0]
change_points_neg = np.where(cusum_neg > threshold)[0]
change_points = np.unique(np.concatenate([change_points_pos, change_points_neg]))

print(f"=== Change Point Detection (CUSUM) ===")
print(f"Threshold: {threshold:.2f}")
print(f"Change points detected: {len(change_points)}")
if len(change_points) > 0:
    print(f"\nChange point indices (first 10):")
    for cp in change_points[:10]:
        ts = df_resampled.index[cp]
        print(f"  Index {cp}: {ts}, Rate = {rate[cp]:.2f}")

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(rate, color='steelblue', linewidth=0.5, alpha=0.7, label='Rate')
axes[0].axhline(mean_val, color='gray', linestyle='--', label=f'Mean: {mean_val:.2f}')
if len(change_points) > 0:
    axes[0].scatter(change_points, rate[change_points], color='red', s=30, zorder=5, label='Change Points')
axes[0].set_title('CUSUM Change Point Detection')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(cusum_pos, color='green', linewidth=1, label='CUSUM+')
axes[1].plot(cusum_neg, color='red', linewidth=1, label='CUSUM-')
axes[1].axhline(threshold, color='orange', linestyle='--', label=f'Threshold: {threshold:.0f}')
axes[1].set_title('CUSUM Statistics')
axes[1].set_ylabel('Cumulative Sum')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/ch9_changepoint.png', dpi=150)
print("\n✅ Change point detection saved to /tmp/ch9_changepoint.png")

---
## Soal 10: Complete Time Series Dashboard

Buat dashboard lengkap yang menggabungkan semua insight time series: plot utama, decomposition, rolling stats, anomalies, dan seasonal pattern.

In [ ]:

fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Panel 1: Main time series
ax1 = fig.add_subplot(gs[0, :2])
rate = df_resampled['Rate'].dropna()
ax1.plot(rate.index, rate.values, alpha=0.5, linewidth=0.5, color='steelblue')
rolling = rate.rolling(window=50, center=True).mean()
ax1.plot(rolling.index, rolling.values, color='red', linewidth=2)
ax1.set_title('Packet Rate Over Time')
ax1.set_ylabel('Rate (pkts/s)')
ax1.grid(True, alpha=0.3)

# Panel 2: Distribution
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(rate.values, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax2.set_title('Distribusi Rate')
ax2.set_xlabel('Rate')

# Panel 3: Rolling stats
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(rate.rolling(50).mean().values, color='blue', linewidth=1, label='Mean')
ax3.plot(rate.rolling(50).std().values, color='orange', linewidth=1, label='Std')
ax3.set_title('Rolling Stats')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Panel 4: ACF
ax4 = fig.add_subplot(gs[1, 1])
try:
    from statsmodels.tsa.stattools import acf
    acf_vals = acf(rate, nlags=50)
    ax4.bar(range(51), acf_vals, color='teal', alpha=0.7)
    ax4.axhline(0.1, color='red', linestyle='--', alpha=0.5)
except:
    ax4.text(0.5, 0.5, 'statsmodels not installed', ha='center', va='center', transform=ax4.transAxes)
ax4.set_title('ACF')
ax4.grid(True, alpha=0.3)

# Panel 5: Anomalies
ax5 = fig.add_subplot(gs[1, 2])
z = (rate - rate.rolling(50, center=True).mean()) / rate.rolling(50, center=True).std()
anom_idx = np.where(np.abs(z.dropna()) > 3)[0]
ax5.scatter(range(len(rate)), rate.values, alpha=0.2, s=5, color='gray')
if len(anom_idx) > 0:
    ax5.scatter(anom_idx, rate.values[anom_idx], color='red', s=20, zorder=5)
ax5.set_title(f'Anomalies: {len(anom_idx)} detected')

# Panel 6: Traffic type over time
ax6 = fig.add_subplot(gs[2, 0])
type_colors = df_resampled['traffic_type'].map({'Benign': 'green', 'Malicious': 'red'})
ax6.scatter(range(len(df_resampled)), df_resampled['Rate'].values,
            c=type_colors.values, alpha=0.3, s=10)
ax6.set_title('Rate by Traffic Type')
ax6.set_xlabel('Time Index')

# Panel 7: Seasonal pattern
ax7 = fig.add_subplot(gs[2, 1])
hourly = df_ts_copy.groupby('hour')['Rate'].mean()
ax7.bar(hourly.index, hourly.values, color='coral', alpha=0.7)
ax7.set_title('Hourly Pattern')
ax7.set_xlabel('Hour')

# Panel 8: Summary stats
ax8 = fig.add_subplot(gs[2, 2])
ax8.axis('off')
stats_text = f"""Time Series Summary
━━━━━━━━━━━━━━━━━━━━━
Data points: {len(rate):,}
Time range: {df_resampled.index[0].strftime('%H:%M:%S')} - {df_resampled.index[-1].strftime('%H:%M:%S')}

Rate Statistics:
  Mean:  {rate.mean():.2f}
  Std:   {rate.std():.2f}
  Min:   {rate.min():.2f}
  Max:   {rate.max():.2f}
  Median: {rate.median():.2f}

Anomalies: {len(anom_idx)} ({len(anom_idx)/len(rate)*100:.1f}%)
Change Points: {len(change_points) if len(change_points) > 0 else 0}"""
ax8.text(0.05, 0.95, stats_text, transform=ax8.transAxes, fontsize=10,
         verticalalignment='top', fontfamily='monospace')

plt.suptitle('Mirai Botnet — Time Series Analysis Dashboard', fontsize=16, y=0.98)
plt.savefig('/tmp/ch9_dashboard.png', dpi=150, bbox_inches='tight')
print("✅ Time Series Dashboard saved to /tmp/ch9_dashboard.png")